# Test dei 4 modelli addestrati (NSynth, rete neurale)

Carica i **pesi addestrati** (`pesi_net.npz`, 4 modelli: Dynamic GD, Newton-CG,
Newton-CG L1, BB-CCV — addestrati localmente con `MAX_ITER=200`, cap batch 2048)
e lo **scaler** (`scaler_net.npz`, RobustScaler 5–95 dei dati di training) e
verifica le previsioni su una **clip reale di uno strumento**.

**File da caricare** (dalla tua cartella locale):
- `pesi_net.npz` (pesi finali)
- `scaler_net.npz` (normalizzazione features)

*(facoltativo, per l'accuratezza sull'intero test set): `nsynth-test/features_opt_net.npz`*

In [ ]:
#@title 0. Dipendenze e import
%pip install -q librosa scikit-learn
import numpy as np
import librosa
import json, io, os
print("librosa", librosa.__version__, "| numpy", np.__version__)

In [ ]:
#@title 1. Carica i pesi addestrati e lo scaler (upload da locale)
from google.colab import files
print("Carica pesi_net.npz (pesi dei 4 modelli):")
f1 = files.upload()     # seleziona pesi_net.npz
print("\nCarica scaler_net.npz (normalizzazione features):")
f2 = files.upload()     # seleziona scaler_net.npz

PESI = np.load(list(f1.keys())[0])
SCAL = np.load(list(f2.keys())[0])
CENTER, SCALE = SCAL["center"], SCAL["scale"]

DISPLAY = {"Dynamic_GD": "Dynamic GD", "Newton-CG": "Newton-CG",
           "Newton-CG_L1": "Newton-CG L1", "BB-CCV": "BB-CCV"}
print("\nModelli caricati:", list(DISPLAY.keys()))
print("Pesi shape:", {k: PESI[k].shape for k in PESI.files})

In [ ]:
#@title 2. Rete neurale (identica al notebook nsynth_net_riproduzione)
SR, N_MELS, HOP, N_FFT = 16000, 96, 256, 1024
N_MFCC, N_CQT = 40, 84
input_dim, h1, h2, out_dim = 780, 256, 128, 10
FAMILIES = ['bass', 'brass', 'flute', 'guitar', 'keyboard',
            'mallet', 'organ', 'reed', 'string', 'vocal']

def param_slices(input_dim, h1, h2, out_dim):
    idx = 0
    W1_s = slice(idx, idx + input_dim*h1); idx += input_dim*h1
    b1_s = slice(idx, idx + h1);           idx += h1
    W2_s = slice(idx, idx + h1*h2);        idx += h1*h2
    b2_s = slice(idx, idx + h2);           idx += h2
    W3_s = slice(idx, idx + h2*out_dim);   idx += h2*out_dim
    b3_s = slice(idx, idx + out_dim);      idx += out_dim
    return W1_s, b1_s, W2_s, b2_s, W3_s, b3_s
W1_s, b1_s, W2_s, b2_s, W3_s, b3_s = param_slices(input_dim, h1, h2, out_dim)

def forward(w, Xb):
    W1 = w[W1_s].reshape(input_dim, h1); b1 = w[b1_s]
    W2 = w[W2_s].reshape(h1, h2);        b2 = w[b2_s]
    W3 = w[W3_s].reshape(h2, out_dim);   b3 = w[b3_s]
    z1 = Xb @ W1 + b1; a1 = np.tanh(z1)
    z2 = a1 @ W2 + b2; a2 = np.tanh(z2)
    logits = a2 @ W3 + b3
    return logits, a1, a2

def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    P = np.exp(Z); return P / P.sum(axis=1, keepdims=True)

In [ ]:
#@title 3. Features audio 780-dim (identiche al notebook)
def estrai_features(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    eps = 1e-10
    y_h, y_p = librosa.effects.hpss(y)
    S    = librosa.feature.melspectrogram(y=y_h, sr=SR, n_mels=N_MELS,
                                          n_fft=N_FFT, hop_length=HOP)
    logS = librosa.power_to_db(S, ref=np.max)
    dS   = librosa.feature.delta(logS)
    d2S  = librosa.feature.delta(logS, order=2)
    f = [logS.mean(1), logS.std(1), dS.std(1), d2S.std(1)]
    mfcc  = librosa.feature.mfcc(y=y_h, sr=SR, n_mfcc=N_MFCC,
                                 n_fft=N_FFT, hop_length=HOP)
    dmfcc = librosa.feature.delta(mfcc)
    f += [mfcc.mean(1), mfcc.std(1), dmfcc.mean(1), dmfcc.std(1)]
    C_cqt = np.abs(librosa.cqt(y_h, sr=SR, hop_length=HOP, n_bins=N_CQT))
    logC  = librosa.amplitude_to_db(C_cqt, ref=np.max)
    f += [logC.mean(1), logC.std(1)]
    tn = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=SR)
    f += [tn.mean(1), tn.std(1)]
    ct = librosa.feature.spectral_contrast(y=y_h, sr=SR, n_fft=N_FFT, hop_length=HOP)
    ch = librosa.feature.chroma_stft(y=y_h, sr=SR, n_fft=N_FFT, hop_length=HOP)
    f += [ct.mean(1), ct.std(1), ch.mean(1), ch.std(1)]
    for v in (librosa.feature.spectral_centroid(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP),
              librosa.feature.spectral_bandwidth(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP),
              librosa.feature.spectral_rolloff(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP),
              librosa.feature.spectral_flatness(y=y, n_fft=N_FFT, hop_length=HOP),
              librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP)):
        f += [v.mean(1), v.std(1)]
    ons = librosa.onset.onset_strength(y=y, sr=SR, hop_length=HOP)
    Eh, Ep = float(np.sum(y_h ** 2)), float(np.sum(y_p ** 2))
    rms  = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP)[0]
    tt   = np.arange(len(rms))
    tc   = float(np.sum(tt * rms) / (np.sum(rms) + eps)) / max(len(rms) - 1, 1)
    f.append(np.array([ons.mean(), ons.std(), ons.max(),
                       np.log((Eh + eps) / (Ep + eps)),
                       np.log((Ep + eps) / (Eh + Ep + eps)),
                       float(rms.max() / (rms.mean() + eps)),
                       tc, float(np.log1p(np.sum(y ** 2)))]))
    return np.concatenate(f).astype(np.float32)

def normalizza(x):
    x = np.atleast_2d((x - CENTER) / SCALE)  # RobustScaler (5-95)
    x = np.clip(x, -10, 10)                  # clip come nel notebook
    return np.nan_to_num(x).astype(np.float32)

In [ ]:
#@title 4. Test su una clip reale (upload .wav)
from google.colab import files
print("Carica una clip .wav di uno strumento (es. da NSynth test):")
f3 = files.upload()
wav = list(f3.keys())[0]

x = normalizza(estrai_features(wav))
print(f"\nClip: {wav}  |  features: {x.shape[1]} dim\n")
print(f"{'Modello':14s} {'Predizione':10s}  {'P (pred)':>8s}   top-3")
print("-" * 56)
for key, disp in DISPLAY.items():
    logits, _, _ = forward(PESI[key], x)
    P = softmax(logits)[0]
    pred = int(np.argmax(P))
    top3 = sorted(zip(FAMILIES, P), key=lambda t: -t[1])[:3]
    s3 = ", ".join(f"{f}:{p:.2f}" for f, p in top3)
    print(f"{disp:14s} {FAMILIES[pred]:10s}  {P[pred]:8.1%}   {s3}")
print()
print("Famiglie possibili:", ", ".join(FAMILIES))

In [ ]:
#@title 5. (Facoltativo) Ascolta la clip caricata
from IPython.display import Audio, display
display(Audio(wav))

In [ ]:
#@title 6. (Facoltativo) Accuratezza sul test set NSynth
# Carica features_opt_net.npz (split test, upload) -> accuratezza dei 4 modelli
from google.colab import files
print("Carica nsynth-test/features_opt_net.npz (X, y, names):")
f4 = files.upload()
z = np.load(list(f4.keys())[0])
X, y = z["X"], z["y"]
X = normalizza(X.astype(np.float32))   # usa lo scaler del training
print(f"Test: {X.shape[0]} clip\n")
print(f"{'Modello':14s} {'Acc. test':>9s}")
print("-" * 28)
for key, disp in DISPLAY.items():
    logits, _, _ = forward(PESI[key], X)
    acc = float(np.mean(np.argmax(softmax(logits), axis=1) == y))
    print(f"{disp:14s} {acc*100:8.1f}%")